# Idea 5: Merced Climate Robustness Analysis

This notebook analyzes the first climate-robustness experiment for the Merced basin. It compares the existing Merced operating rules under historical Livneh hydrology, GCM ACCESS1_0_rcp85 hydrology, and LOCA2 ACCESS_CM2 hydrology. The notebook is designed to be self-contained: it discovers the model result folders, loads the Pywr output CSVs, calculates annual and seasonal metrics, generates figures, and saves publication-ready PNG files.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ImportError:
    sns = None

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_ROOT = Path.cwd()
RESULTS_ROOT = next((p for p in [PROJECT_ROOT / "RESULTS_", PROJECT_ROOT / "results"] if p.exists()), None)
if RESULTS_ROOT is None:
    raise FileNotFoundError("Could not find a RESULTS_ or results folder in the current project directory.")

FIG_DIR = PROJECT_ROOT / "figures" / "idea5_merced"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if sns is not None:
    sns.set_theme(style="whitegrid", context="talk")
else:
    plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

PALETTE = {
    "Historical Livneh": "#2F6B4F",
    "GCM ACCESS1-0 RCP8.5": "#B65D2E",
    "LOCA2 ACCESS-CM2": "#3C6EAA",
}

MONTH_ORDER = [10, 11, 12, 1, 2, 3, 4, 5, 6, 7, 8, 9]
MONTH_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Using results root: {RESULTS_ROOT}")
print(f"Figures will be saved to: {FIG_DIR}")

## Discover Merced Result Folders

The future runs have flat CSV headers. The historical run was executed with the `natural_flow` scenario set, so its CSVs include both `Baseline` and `natural_flow` scenario columns. For the main climate-robustness comparison, this notebook uses the `Baseline` scenario from the historical run. Change `historical_scenario` below if a different historical scenario is desired.

In [ ]:
def latest_match(pattern):
    matches = sorted(RESULTS_ROOT.glob(pattern), key=lambda p: p.stat().st_mtime)
    if not matches:
        raise FileNotFoundError(f"No result folder matched pattern: {pattern}")
    return matches[-1]

historical_scenario = "Baseline"

RUNS = {
    "Historical Livneh": {
        "path": latest_match("Natural Flow*/merced/historical/Livneh"),
        "climate_set": "historical",
        "climate_model": "Livneh",
        "hydrology_path": PROJECT_ROOT / "data/Merced_River/hydrology/historical/Livneh/preprocessed/full_natural_flow_daily_mcm.csv",
        "scenario": historical_scenario,
    },
    "GCM ACCESS1-0 RCP8.5": {
        "path": latest_match("GCMs test*/merced/gcms/ACCESS1_0_rcp85"),
        "climate_set": "gcms",
        "climate_model": "ACCESS1_0_rcp85",
        "hydrology_path": PROJECT_ROOT / "data/Merced_River/hydrology/gcms/ACCESS1_0_rcp85/preprocessed/full_natural_flow_daily_mcm.csv",
        "scenario": None,
    },
    "LOCA2 ACCESS-CM2": {
        "path": latest_match("LOCA2_GCMs test*/merced/LOCA2_gcms/ACCESS_CM2"),
        "climate_set": "LOCA2_gcms",
        "climate_model": "ACCESS_CM2",
        "hydrology_path": PROJECT_ROOT / "data/Merced_River/hydrology/LOCA2_gcms/ACCESS_CM2/preprocessed/full_natural_flow_daily_mcm.csv",
        "scenario": None,
    },
}

for label, cfg in RUNS.items():
    print(f"{label}: {cfg['path']}")

## Helper Functions

In [ ]:
def water_year(index):
    idx = pd.DatetimeIndex(index)
    return idx.year + (idx.month >= 10).astype(int)


def read_pywr_csv(path, scenario=None):
    """Read either a flat Pywr output CSV or a scenario-expanded Pywr CSV."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8-sig") as f:
        first_line = f.readline().strip()

    if first_line.startswith("node,"):
        df = pd.read_csv(path, header=[0, 1, 2], index_col=0, parse_dates=True)
        df.index.name = "Date"
        df.columns = pd.MultiIndex.from_tuples(
            [(str(a), str(b), str(c)) for a, b, c in df.columns],
            names=["node", "scenario", "unused"],
        )
        if scenario is None:
            scenario = df.columns.get_level_values("scenario")[0]
        available = sorted(set(df.columns.get_level_values("scenario")))
        if scenario not in available:
            raise ValueError(f"Scenario {scenario!r} not found in {path.name}. Available: {available}")
        df = df.xs(scenario, level="scenario", axis=1, drop_level=False)
        df.columns = df.columns.get_level_values("node")
    else:
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        df.index.name = "Date"

    df = df.apply(pd.to_numeric, errors="coerce")
    return df.sort_index()


def read_run_file(run_label, filename):
    cfg = RUNS[run_label]
    return read_pywr_csv(cfg["path"] / filename, scenario=cfg.get("scenario"))


def read_hydrology(run_label):
    cfg = RUNS[run_label]
    df = pd.read_csv(cfg["hydrology_path"], index_col=0, parse_dates=True)
    df.index.name = "Date"
    if "flow" not in df.columns:
        df.columns = ["flow"]
    return df[["flow"]].rename(columns={"flow": run_label}).sort_index()


def annual_sum(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out.groupby("Water Year").sum(numeric_only=True)


def annual_mean(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out.groupby("Water Year").mean(numeric_only=True)


def annual_min(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out.groupby("Water Year").min(numeric_only=True)


def seasonal_monthly_profile(series_or_df, agg="mean"):
    df = series_or_df.to_frame() if isinstance(series_or_df, pd.Series) else series_or_df.copy()
    monthly = df.resample("MS").mean()
    monthly["month"] = monthly.index.month
    if agg == "median":
        profile = monthly.groupby("month").median(numeric_only=True)
    else:
        profile = monthly.groupby("month").mean(numeric_only=True)
    return profile.reindex(MONTH_ORDER)


def add_water_year_column(df):
    out = df.copy()
    out["Water Year"] = water_year(out.index)
    return out


def savefig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")
    return path

## Load Model Outputs

In [ ]:
FILES = {
    "storage": "Reservoir_Storage_mcm.csv",
    "elevation": "Reservoir_Elevation_m.csv",
    "hydropower_energy": "Hydropower_Energy_MWh.csv",
    "hydropower_flow": "Hydropower_Flow_mcm.csv",
    "ifr_flow": "InstreamFlowRequirement_Flow_mcm.csv",
    "ifr_min": "InstreamFlowRequirement_Min Flow_mcm.csv",
    "output_flow": "Output_Flow_mcm.csv",
    "output_demand": "Output_Demand_mcm.csv",
    "flood_release": "PiecewiseLink_Flow_mcm.csv",
}

data = {label: {} for label in RUNS}
for label in RUNS:
    for key, filename in FILES.items():
        f = RUNS[label]["path"] / filename
        if f.exists():
            data[label][key] = read_run_file(label, filename)
    data[label]["hydrology"] = read_hydrology(label)

for label, sections in data.items():
    print(f"\n{label}")
    for key, df in sections.items():
        print(f"  {key:18s} {df.shape[0]:6d} rows x {df.shape[1]:2d} cols | {df.index.min().date()} to {df.index.max().date()}")

## Calculate Summary Metrics

In [ ]:
summary_rows = []
annual_tables = {}

for label, sections in data.items():
    annual_energy = annual_sum(sections["hydropower_energy"]).sum(axis=1)
    annual_outflow = annual_sum(sections["output_flow"][["Merced River Outflow"]]).iloc[:, 0]
    annual_runoff = annual_sum(sections["hydrology"]).iloc[:, 0]
    storage = sections["storage"]["Lake McClure"]
    annual_storage_mean = annual_mean(storage.to_frame("Lake McClure")).iloc[:, 0]
    annual_storage_min = annual_min(storage.to_frame("Lake McClure")).iloc[:, 0]

    demand = sections["output_demand"]
    delivery = sections["output_flow"][demand.columns.intersection(sections["output_flow"].columns)]
    demand = demand[delivery.columns]
    shortage = (demand - delivery).clip(lower=0)
    annual_demand = annual_sum(demand).sum(axis=1)
    annual_shortage = annual_sum(shortage).sum(axis=1)
    annual_delivery_reliability = 1 - annual_shortage.divide(annual_demand.replace(0, np.nan))

    ifr_min = sections["ifr_min"]
    ifr_flow = sections["ifr_flow"][ifr_min.columns.intersection(sections["ifr_flow"].columns)]
    ifr_min = ifr_min[ifr_flow.columns]
    ifr_deficit = (ifr_min - ifr_flow).clip(lower=0)
    annual_ifr_requirement = annual_sum(ifr_min).sum(axis=1)
    annual_ifr_deficit = annual_sum(ifr_deficit).sum(axis=1)
    annual_ifr_reliability = 1 - annual_ifr_deficit.divide(annual_ifr_requirement.replace(0, np.nan))

    flood = sections.get("flood_release")
    annual_flood_release = annual_sum(flood).sum(axis=1) if flood is not None else pd.Series(dtype=float)

    annual = pd.DataFrame({
        "runoff_mcm": annual_runoff,
        "hydropower_mwh": annual_energy,
        "outflow_mcm": annual_outflow,
        "mean_lake_mcclure_storage_mcm": annual_storage_mean,
        "min_lake_mcclure_storage_mcm": annual_storage_min,
        "delivery_demand_mcm": annual_demand,
        "delivery_shortage_mcm": annual_shortage,
        "delivery_reliability": annual_delivery_reliability,
        "ifr_requirement_mcm": annual_ifr_requirement,
        "ifr_deficit_mcm": annual_ifr_deficit,
        "ifr_reliability": annual_ifr_reliability,
        "flood_release_mcm": annual_flood_release,
    })
    annual["Scenario"] = label
    annual_tables[label] = annual

    summary_rows.append({
        "Scenario": label,
        "Start": sections["storage"].index.min().date(),
        "End": sections["storage"].index.max().date(),
        "Water Years": annual.index.min().astype(int) if hasattr(annual.index.min(), "astype") else int(annual.index.min()),
        "Mean annual runoff (mcm/yr)": annual_runoff.mean(),
        "Mean annual hydropower (MWh/yr)": annual_energy.mean(),
        "Mean annual outflow (mcm/yr)": annual_outflow.mean(),
        "Mean Lake McClure storage (mcm)": storage.mean(),
        "Minimum Lake McClure storage (mcm)": storage.min(),
        "Mean delivery reliability": annual_delivery_reliability.mean(),
        "Mean IFR reliability": annual_ifr_reliability.mean(),
        "Mean annual flood release proxy (mcm/yr)": annual_flood_release.mean() if len(annual_flood_release) else np.nan,
    })

annual_metrics = pd.concat(annual_tables.values()).reset_index().rename(columns={"index": "Water Year"})
summary = pd.DataFrame(summary_rows)
summary_path = FIG_DIR / "merced_climate_robustness_summary.csv"
annual_path = FIG_DIR / "merced_climate_robustness_annual_metrics.csv"
summary.to_csv(summary_path, index=False)
annual_metrics.to_csv(annual_path, index=False)

display(summary.round(3))
print(f"Saved summary: {summary_path}")
print(f"Saved annual metrics: {annual_path}")

## Figure 1: Seasonal Full Natural Flow Forcing

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))

for label in RUNS:
    profile = seasonal_monthly_profile(data[label]["hydrology"])
    ax.plot(MONTH_LABELS, profile[label].values, marker="o", linewidth=2.5, color=PALETTE[label], label=label)

ax.set_title("Merced Basin Hydrologic Forcing")
ax.set_ylabel("Mean full natural flow (mcm/day)")
ax.set_xlabel("Water-year month")
ax.legend(frameon=False, ncol=1)
ax.grid(True, alpha=0.25)
fig.tight_layout()
savefig(fig, "fig01_merced_hydrology_seasonal_profile.png")
plt.show()

## Figure 2: Lake McClure Seasonal Storage Profile

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))

for label in RUNS:
    storage = data[label]["storage"][["Lake McClure"]]
    profile = seasonal_monthly_profile(storage)
    ax.plot(MONTH_LABELS, profile["Lake McClure"].values, marker="o", linewidth=2.5, color=PALETTE[label], label=label)

ax.set_title("Lake McClure Seasonal Storage Under Existing Rules")
ax.set_ylabel("Mean storage (mcm)")
ax.set_xlabel("Water-year month")
ax.legend(frameon=False)
ax.grid(True, alpha=0.25)
fig.tight_layout()
savefig(fig, "fig02_lake_mcclure_storage_profile.png")
plt.show()

## Figure 3: Annual Hydropower Generation

In [ ]:
plot_df = annual_metrics[["Scenario", "Water Year", "hydropower_mwh"]].copy()

fig, ax = plt.subplots(figsize=(9.5, 5.2))
if sns is not None:
    sns.boxplot(data=plot_df, x="Scenario", y="hydropower_mwh", palette=PALETTE, ax=ax, width=0.55, fliersize=0)
    sns.stripplot(data=plot_df, x="Scenario", y="hydropower_mwh", palette=PALETTE, ax=ax, size=4, alpha=0.55, jitter=0.18)
else:
    groups = [plot_df.loc[plot_df["Scenario"] == s, "hydropower_mwh"].dropna() for s in RUNS]
    ax.boxplot(groups, labels=list(RUNS), showfliers=False)

ax.set_title("Annual Hydropower Generation")
ax.set_ylabel("Hydropower generation (MWh/year)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=12)
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
savefig(fig, "fig03_annual_hydropower_generation.png")
plt.show()

## Figure 4: Delivery Reliability And Shortage

In [ ]:
delivery_summary = annual_metrics.groupby("Scenario", as_index=False).agg(
    mean_reliability=("delivery_reliability", "mean"),
    mean_shortage=("delivery_shortage_mcm", "mean"),
)
delivery_summary["mean_reliability_pct"] = 100 * delivery_summary["mean_reliability"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=False)

axes[0].bar(delivery_summary["Scenario"], delivery_summary["mean_reliability_pct"], color=[PALETTE[s] for s in delivery_summary["Scenario"]])
axes[0].set_title("Delivery Reliability")
axes[0].set_ylabel("Mean reliability (%)")
axes[0].set_ylim(0, 105)
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(delivery_summary["Scenario"], delivery_summary["mean_shortage"], color=[PALETTE[s] for s in delivery_summary["Scenario"]])
axes[1].set_title("Delivery Shortage")
axes[1].set_ylabel("Mean shortage (mcm/year)")
axes[1].tick_params(axis="x", rotation=15)

for ax in axes:
    ax.set_xlabel("")
    ax.grid(True, axis="y", alpha=0.25)

fig.suptitle("MID Delivery Performance", y=1.03, fontsize=15, fontweight="bold")
fig.tight_layout()
savefig(fig, "fig04_delivery_reliability_and_shortage.png")
plt.show()

display(delivery_summary.round(3))

## Figure 5: Instream Flow Reliability

In [ ]:
ifr_daily_rows = []
for label in RUNS:
    ifr_min = data[label]["ifr_min"]
    ifr_flow = data[label]["ifr_flow"][ifr_min.columns.intersection(data[label]["ifr_flow"].columns)]
    ifr_min = ifr_min[ifr_flow.columns]
    deficit = (ifr_min - ifr_flow).clip(lower=0)
    for node in deficit.columns:
        tmp = pd.DataFrame({
            "Scenario": label,
            "Node": node,
            "Deficit": deficit[node].values,
            "Requirement": ifr_min[node].values,
            "Date": deficit.index,
        })
        ifr_daily_rows.append(tmp)

ifr_daily = pd.concat(ifr_daily_rows, ignore_index=True)
ifr_summary = ifr_daily.groupby(["Scenario", "Node"], as_index=False).agg(
    total_deficit_mcm=("Deficit", "sum"),
    total_requirement_mcm=("Requirement", "sum"),
    deficit_days=("Deficit", lambda s: int((s > 1e-8).sum())),
)
ifr_summary["reliability_pct"] = 100 * (1 - ifr_summary["total_deficit_mcm"] / ifr_summary["total_requirement_mcm"].replace(0, np.nan))

fig, ax = plt.subplots(figsize=(10, 5.4))
if sns is not None:
    sns.barplot(data=ifr_summary, x="Node", y="reliability_pct", hue="Scenario", palette=PALETTE, ax=ax)
else:
    pivot = ifr_summary.pivot(index="Node", columns="Scenario", values="reliability_pct")
    pivot.plot(kind="bar", ax=ax, color=[PALETTE[s] for s in pivot.columns])

ax.set_title("Instream Flow Reliability")
ax.set_ylabel("Reliability (%)")
ax.set_xlabel("")
ax.set_ylim(0, 105)
ax.tick_params(axis="x", rotation=10)
ax.legend(frameon=False, title="")
ax.grid(True, axis="y", alpha=0.25)
fig.tight_layout()
savefig(fig, "fig05_instream_flow_reliability.png")
plt.show()

display(ifr_summary.round(3))

## Figure 6: Merced River Outflow Flow-Duration Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))

for label in RUNS:
    series = data[label]["output_flow"]["Merced River Outflow"].dropna().sort_values(ascending=False).reset_index(drop=True)
    exceedance = 100 * (np.arange(1, len(series) + 1) / (len(series) + 1))
    ax.plot(exceedance, series.values, linewidth=2.2, color=PALETTE[label], label=label)

ax.set_title("Merced River Outflow Duration Curves")
ax.set_xlabel("Exceedance probability (%)")
ax.set_ylabel("Daily outflow (mcm/day)")
ax.set_yscale("symlog", linthresh=0.01)
ax.legend(frameon=False)
ax.grid(True, which="both", alpha=0.25)
fig.tight_layout()
savefig(fig, "fig06_merced_outflow_duration_curves.png")
plt.show()

## Figure 7: Flood Release Proxy

In [ ]:
flood_rows = []
for label in RUNS:
    flood = data[label].get("flood_release")
    if flood is None or flood.empty:
        continue
    tmp = annual_sum(flood).sum(axis=1).rename("flood_release_mcm").reset_index()
    tmp["Scenario"] = label
    flood_rows.append(tmp)

if flood_rows:
    flood_annual = pd.concat(flood_rows, ignore_index=True)
    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    if sns is not None:
        sns.boxplot(data=flood_annual, x="Scenario", y="flood_release_mcm", palette=PALETTE, ax=ax, width=0.55, fliersize=0)
        sns.stripplot(data=flood_annual, x="Scenario", y="flood_release_mcm", palette=PALETTE, ax=ax, size=4, alpha=0.55, jitter=0.18)
    else:
        groups = [flood_annual.loc[flood_annual["Scenario"] == s, "flood_release_mcm"].dropna() for s in RUNS]
        ax.boxplot(groups, labels=list(RUNS), showfliers=False)
    ax.set_title("Exchequer Dam Flood Release Proxy")
    ax.set_ylabel("Annual release through flood-release link (mcm/year)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=12)
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()
    savefig(fig, "fig07_flood_release_proxy.png")
    plt.show()
    display(flood_annual.groupby("Scenario")["flood_release_mcm"].describe().round(3))
else:
    print("No flood-release recorder found in the loaded results.")

## Figure 8: Robustness Summary Heatmap

In [ ]:
metric_display = annual_metrics.groupby("Scenario").agg({
    "runoff_mcm": "mean",
    "hydropower_mwh": "mean",
    "delivery_reliability": "mean",
    "ifr_reliability": "mean",
    "mean_lake_mcclure_storage_mcm": "mean",
    "min_lake_mcclure_storage_mcm": "mean",
    "flood_release_mcm": "mean",
})

metric_display = metric_display.rename(columns={
    "runoff_mcm": "Runoff",
    "hydropower_mwh": "Hydropower",
    "delivery_reliability": "Delivery reliability",
    "ifr_reliability": "IFR reliability",
    "mean_lake_mcclure_storage_mcm": "Mean storage",
    "min_lake_mcclure_storage_mcm": "Mean annual min storage",
    "flood_release_mcm": "Flood release proxy",
})

baseline = metric_display.loc["Historical Livneh"].replace(0, np.nan)
relative = 100 * (metric_display.divide(baseline) - 1)
relative.loc["Historical Livneh"] = 0

fig, ax = plt.subplots(figsize=(10.5, 4.8))
if sns is not None:
    sns.heatmap(relative, annot=True, fmt=".1f", cmap="RdBu", center=0, linewidths=0.5, cbar_kws={"label": "% change vs historical"}, ax=ax)
else:
    im = ax.imshow(relative.values, cmap="RdBu", vmin=-np.nanmax(abs(relative.values)), vmax=np.nanmax(abs(relative.values)))
    ax.set_xticks(range(relative.shape[1]), relative.columns, rotation=35, ha="right")
    ax.set_yticks(range(relative.shape[0]), relative.index)
    fig.colorbar(im, ax=ax, label="% change vs historical")

ax.set_title("Climate Robustness Metrics Relative to Historical")
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
savefig(fig, "fig08_robustness_summary_heatmap.png")
plt.show()

display(relative.round(2))

## Interpretation Notes

Use these figures to assess whether the existing Merced operating rules are robust across hydrologic futures. Evidence of reduced robustness would include lower delivery reliability, larger IFR deficits, lower carryover storage, more frequent flood releases, or a large shift in hydropower generation relative to the historical case. Because the historical run uses a shorter period than the future runs, the comparison should emphasize annualized metrics rather than total-period sums.